In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.util, json, shutil, subprocess, sys, time, traceback
import numpy as np
import pandas as pd

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
DATA = WORK / 'day10-cbs-data'
PROJECT = WORK / 'day10-cbs-project-src'

# Locate the largest attached checkpoint with aligned feature and sequence files.
data_candidates = []
for feature_path in INPUT.rglob('features.csv'):
    root = feature_path.parent
    if not all((root / name).is_file() for name in ('manifest.csv', 'series.npy')):
        continue
    try:
        frame = pd.read_csv(feature_path, usecols=['source_id'], dtype={'source_id': 'string'})
        tic = frame['source_id'].astype('string').str.fullmatch(r'\d{8,16}', na=False)
        data_candidates.append((len(frame), int(frame.loc[tic, 'source_id'].nunique()), root))
    except Exception as exc:
        print('Skipping unreadable candidate:', root, type(exc).__name__, str(exc)[:180])
if not data_candidates:
    raise FileNotFoundError('Attach the successful 20K notebook output containing cbs-data/manifest.csv, features.csv, and series.npy.')
rows, unique_tics, DATA_SOURCE = max(data_candidates, key=lambda item: (item[0], item[1], str(item[2])))
if rows < 19_000:
    raise RuntimeError(f'Largest attached feature table has {rows:,} rows, below the 19,000-row checkpoint guard. Attach Cosmic 20000 swan v2 output.')

project_candidates = []
for config in INPUT.rglob('pyproject.toml'):
    root = config.parent
    if (root / 'src/cosmic_black_swan/cli.py').is_file():
        project_candidates.append(root)
if not project_candidates:
    raise FileNotFoundError('Attach Cosmic Swan 4 output containing cbs-project-src/pyproject.toml and src/cosmic_black_swan/.')
PROJECT_SOURCE = max(project_candidates, key=lambda p: len(list((p / 'src/cosmic_black_swan').glob('*.py'))))

# Make writable copies; retain any existing labels, model scores and run metadata.
DATA.mkdir(parents=True, exist_ok=True)
for name in ('manifest.csv', 'features.csv', 'series.npy', 'labels.csv', 'feature_errors.csv', 'day6_run_summary.json'):
    src = DATA_SOURCE / name
    if src.exists():
        shutil.copy2(src, DATA / name)
for name in ('scores', 'models', 'catalogs'):
    src = DATA_SOURCE / name
    if src.is_dir():
        shutil.copytree(src, DATA / name, dirs_exist_ok=True, ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))
shutil.copytree(PROJECT_SOURCE, PROJECT, dirs_exist_ok=True, ignore=shutil.ignore_patterns('__pycache__', '*.pyc', '.git'))
sys.path.insert(0, str(PROJECT / 'src'))

features = pd.read_csv(DATA / 'features.csv', dtype={'source_id': 'string'})
series = np.load(DATA / 'series.npy', mmap_mode='r')
if len(features) != len(series):
    raise ValueError(f'Feature/sequence row mismatch: {len(features)} vs {len(series)}')
manifest = pd.read_csv(DATA / 'manifest.csv', usecols=['status'])
completed = int(manifest['status'].astype(str).isin(['COMPLETE', 'ARCHIVED']).sum())
valid_tics = features['source_id'].astype('string').str.fullmatch(r'\d{8,16}', na=False)
print('Selected checkpoint:', DATA_SOURCE)
print('Project source:', PROJECT_SOURCE)
print(f'Checkpoint rows: {len(features):,}; unique TICs: {features.loc[valid_tics, "source_id"].nunique():,}; manifest products: {completed:,}')
print('Sequence shape:', tuple(series.shape), '| dtype:', series.dtype)
print('Writable data:', DATA)
print('PyTorch available:', importlib.util.find_spec('torch') is not None)
print('XGBoost available:', importlib.util.find_spec('xgboost') is not None)

# Keep the installed Kaggle CUDA/PyTorch stack intact. Install only XGBoost if absent.
if importlib.util.find_spec('xgboost') is None:
    print('Installing XGBoost package only...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost>=2.0'], check=True)


In [ ]:
STAGE_LOG = []
SCORES = DATA / 'scores'
SCORES.mkdir(parents=True, exist_ok=True)

def read_json(path):
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else None

def stage(name, fn):
    started = time.time()
    print(f'\n===== {name} =====', flush=True)
    try:
        result = fn()
        row = {'stage': name, 'status': 'complete', 'seconds': round(time.time() - started, 1), 'result': result}
    except Exception as exc:
        row = {'stage': name, 'status': 'failed', 'seconds': round(time.time() - started, 1), 'error': f'{type(exc).__name__}: {exc}', 'traceback': traceback.format_exc(limit=6)}
    STAGE_LOG.append(row)
    print(json.dumps(row, indent=2, default=str), flush=True)
    return row

# XGBoost: reuse its prior source-disjoint result, or build traceable catalog labels and fit once.
def xgb_stage():
    saved = read_json(SCORES / 'xgboost_run.json')
    if saved and saved.get('metrics'):
        return {'action': 'reused_existing_source_disjoint_run', 'run': saved}
    labels_path = DATA / 'labels.csv'
    if not labels_path.exists():
        print('No saved labels.csv; attempting Gaia DR3 / TESS-EB catalog cross-match. Internet access is required.', flush=True)
        from cosmic_black_swan.catalog_labels import build_catalog_labels
        label_run = build_catalog_labels(DATA, match_radius_arcsec=2.0, minimum_gaia_score=0.8)
        print('Catalog labeling summary:', json.dumps(label_run, indent=2))
    if not labels_path.exists():
        return {'action': 'blocked_no_catalog_labels'}
    labels = pd.read_csv(labels_path, dtype={'source_id': 'string', 'class_label': 'string'})
    supported = labels.dropna(subset=['class_label'])
    supported = supported[supported['class_label'].astype(str).str.strip().ne('')]
    print('Supported labeled TICs:', supported['source_id'].nunique(), '| classes:', supported['class_label'].nunique())
    if supported['source_id'].nunique() < 10 or supported['class_label'].nunique() < 2:
        return {'action': 'blocked_insufficient_supported_catalog_labels', 'labeled_sources': int(supported['source_id'].nunique()), 'classes': int(supported['class_label'].nunique())}
    from cosmic_black_swan.classifier import train_xgboost
    return train_xgboost(DATA, labels_path, seed=42, minimum_evidence='single_catalog')

stage('XGBoost source-disjoint catalog-class evaluation', xgb_stage)

# Transformer: reuse an existing held-out run; otherwise train five epochs on GPU.
def transformer_stage():
    saved = read_json(SCORES / 'transformer_run.json')
    if saved and saved.get('heldout_sources', 0):
        return {'action': 'reused_existing_heldout_run', 'run': saved}
    import torch
    if not torch.cuda.is_available():
        return {'action': 'blocked_gpu_not_available', 'device': 'cpu', 'instruction': 'Select GPU T4 in Kaggle settings, restart the session, and rerun.'}
    from cosmic_black_swan.transformer import train_transformer
    return train_transformer(DATA, epochs=5, batch_size=128, learning_rate=3e-4, seed=42, evaluation_masks=8)

stage('Masked-patch Transformer source-heldout scoring', transformer_stage)

# Keep existing calibration output if present; otherwise create its documented field-relative ranking.
def conformal_stage():
    saved = read_json(SCORES / 'conformal_run.json')
    if saved:
        return {'action': 'reused_existing_field_relative_run', 'run': saved}
    from cosmic_black_swan.conformal import run_conformal_screen
    return run_conformal_screen(DATA, seed=42)

stage('Field-relative conformal ranking audit', conformal_stage)

In [ ]:
# Compact, machine-readable Day 10 evidence summary.
summary = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'input_checkpoint': str(DATA_SOURCE),
    'checkpoint_feature_rows': int(len(features)),
    'checkpoint_unique_tics': int(features.loc[valid_tics, 'source_id'].nunique()),
    'checkpoint_manifest_products': completed,
    'stages': STAGE_LOG,
    'artifacts': {},
    'claim_limits': {
        'xgboost': 'Source-disjoint metrics apply only to supported catalog-labeled sources and the recorded split.',
        'transformer': 'Masked reconstruction residual is an unsupervised ranking score, not a probability or validated event classifier.',
        'conformal': 'Current calibration is drawn from the observed field; the code records fdr_control_claim=false. Do not describe this result as calibrated FDR control.',
        'scope': 'This notebook reuses the saved checkpoint and does not refetch light curves or rerun Day 9 Bayesian sampling.'
    }
}

for name in ('isolation_forest_run.json', 'run.json', 'xgboost_run.json', 'transformer_run.json', 'conformal_run.json', 'injection_recovery_run.json'):
    path = SCORES / name
    if path.exists():
        summary['artifacts'][name] = read_json(path)

# Extract concise model metrics without turning scores into probabilities.
metric_rows = []
xgb = summary['artifacts'].get('xgboost_run.json')
if xgb:
    metrics = xgb.get('metrics', {})
    metric_rows.append({'model': 'XGBoost', 'status': 'available', 'n_train_sources': xgb.get('training_rows'), 'n_test_sources': xgb.get('heldout_sources'), 'accuracy': metrics.get('source_disjoint_test_accuracy'), 'macro_f1': metrics.get('source_disjoint_test_macro_f1'), 'detail': 'Source-disjoint catalog-labeled holdout'})
else:
    metric_rows.append({'model': 'XGBoost', 'status': 'not evaluated', 'detail': 'See stage log; labels or dependency may be unavailable'})

transformer = summary['artifacts'].get('transformer_run.json')
if transformer:
    hist = transformer.get('training_history', [])
    last_loss = hist[-1].get('masked_patch_smooth_l1') if hist else None
    audit = transformer.get('relative_rms_score_audit') or {}
    metric_rows.append({'model': 'Masked-patch Transformer', 'status': 'available', 'n_train_sources': transformer.get('train_unique_sources'), 'n_test_sources': transformer.get('heldout_unique_sources'), 'accuracy': None, 'macro_f1': None, 'detail': f"held-out masked MAE; final train loss={last_loss}; Spearman vs relative RMS={audit.get('spearman_rho')}"})
else:
    metric_rows.append({'model': 'Masked-patch Transformer', 'status': 'not evaluated', 'detail': 'See stage log; GPU or PyTorch may be unavailable'})

conformal = summary['artifacts'].get('conformal_run.json')
if conformal:
    metric_rows.append({'model': 'Conformal field-tail screen', 'status': 'field-relative ranking', 'n_train_sources': conformal.get('training_unique_sources'), 'n_test_sources': conformal.get('test_unique_sources'), 'accuracy': None, 'macro_f1': None, 'detail': f"fdr_control_claim={conformal.get('fdr_control_claim', False)}; {conformal.get('interpretation', '')}"})

OUT = WORK / 'day10-ml-evidence'
OUT.mkdir(parents=True, exist_ok=True)
pd.DataFrame(metric_rows).to_csv(OUT / 'day10_ml_metric_summary.csv', index=False)
(OUT / 'day10_ml_evidence_summary.json').write_text(json.dumps(summary, indent=2, default=str), encoding='utf-8')
report = [
    '# Day 10 ML evidence report',
    '',
    f"- Checkpoint: {len(features):,} feature observations across {int(features.loc[valid_tics, 'source_id'].nunique()):,} unique TICs.",
    f'- Completed or archived manifest products: {completed:,}.',
    '- No light curves were downloaded or refetched by this notebook.',
    '',
    '## Model evidence',
    '',
    '| Model | Status | Train sources | Test sources | Accuracy | Macro-F1 | Detail |',
    '|---|---|---:|---:|---:|---:|---|'
]
for row in metric_rows:
    detail = str(row.get('detail', '')).replace('|', '\\|')
    report.append(f"| {row['model']} | {row['status']} | {row.get('n_train_sources', '')} | {row.get('n_test_sources', '')} | {row.get('accuracy', '')} | {row.get('macro_f1', '')} | {detail} |")
report += ['', '## Interpretation limits', '']
report += [f'- {value}' for value in summary['claim_limits'].values()]
(OUT / 'day10_ml_evidence_report.md').write_text('\n'.join(report) + '\n', encoding='utf-8')
print('Saved outputs:', OUT)
print(pd.DataFrame(metric_rows).to_string(index=False))
print(json.dumps({'checkpoint_rows': len(features), 'unique_tics': int(features.loc[valid_tics, 'source_id'].nunique()), 'stage_statuses': [(r['stage'], r['status']) for r in STAGE_LOG], 'outputs': sorted(p.name for p in OUT.iterdir())}, indent=2))